# Dump the full JSC test set — inference only, no retraining

**What this is for.** Gate 1b (`docs/project-brief.md` §11) requires the bitstream to reproduce
the software model's accuracy over the **whole 166,000-sample test set**, to the sample. The
training notebook only ever saved **1,000** samples, which cannot support that claim.

**Why this is a separate notebook.** Re-running `dwn_jsc_kaggle.ipynb` would retrain from
scratch. Seeds are fixed, but CUDA kernels accumulate in non-deterministic order, so you would
get a *slightly different model* — different tables, different wiring. That would invalidate
every number measured against the current checkpoint: 108 core LUTs, 202 comparators, the
passing Gate 1 runs. This notebook **loads the existing checkpoint and only runs inference**, so
the model is bit-identical to the one already verified.

**Why Kaggle at all, for inference?** Upstream `torch_dwn` has no CPU path — `lut_layer.py`
raises `EFDFunction CPU not Implemented` in forward *and* backward. Even a forward pass needs a
GPU.

---

## Setup, once

1. **Create a Kaggle Dataset** holding the artifacts from `training/artifacts/`:
   - `dwn_jsc_t200_distributive_50_l_b100_checkpoint.pt` (required)
   - `dwn_jsc_t200_distributive_50_l_b100_testvectors.npz` (optional, enables a stronger check)

   *Datasets → New Dataset → upload both files.* Name it anything; the next cell finds it.
2. **Import this notebook**, then **Add Input →** your dataset.
3. Settings panel: **Accelerator → GPU**, **Internet → On** (needed for the clone and the
   OpenML fetch).
4. **Run All.**

Runtime is a few minutes: no training happens here.

## What comes out

`dwn_jsc_<run>_testset_full.npz` — `x_raw` (166000 × 16, scaled), `y` (true labels), `pred`
(the software model's predictions). Download it into `training/artifacts/`. It is gitignored:
tens of MB and regenerable by re-running this.

`x_binarized` is deliberately **not** saved — at 166k × 3200 bits it is ~530 MB and nothing
consumes it, because the board's own thermometer encoder turns features into bits.

In [ ]:
# ---- environment + upstream library ----
# Same pinned commit and the same --no-build-isolation reasoning as the training notebook:
# upstream declares torch as a BUILD requirement, so a plain `pip install .` builds against a
# second torch downloaded from PyPI and the extension fails to load at runtime.
import torch

print('torch     :', torch.__version__)
print('cuda avail:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Set Accelerator -> GPU. torch_dwn has no CPU path, so even '
                     'inference needs one.')
print('gpu       :', torch.cuda.get_device_name(0))

PINNED_COMMIT = '9f887a0b4bd84dabf6d8c9ae35368ab2a7e0e3c0'
!rm -rf /kaggle/working/DWN
!git clone --quiet https://github.com/alanbacellar/DWN.git /kaggle/working/DWN
!cd /kaggle/working/DWN && git checkout --quiet {PINNED_COMMIT} && git log -1 --format='pinned at %h %s'
!cd /kaggle/working/DWN && pip install --no-build-isolation . 2>&1 | tail -3

In [ ]:
# ---- find and load the checkpoint ----
# The checkpoint carries everything needed to rebuild the model EXACTLY: the config, the
# state_dict, the fitted thermometer thresholds and the scaler. None of it is refitted here --
# refitting would silently produce a different model from the one already verified.
import glob, os
import torch, torch_dwn as dwn

cands = sorted(glob.glob('/kaggle/input/**/*_checkpoint.pt', recursive=True))
if not cands:
    raise SystemExit('No *_checkpoint.pt under /kaggle/input. Add your dataset via '
                     '"Add Input" in the right-hand panel.')
CKPT_PATH = cands[0]
print('checkpoint:', CKPT_PATH)
if len(cands) > 1:
    print('NOTE: several checkpoints found, using the first:', cands)

ck = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
CONFIG = ck['config']
RUN_NAME = ck.get('run_name', 'unnamed')

print('run       :', RUN_NAME)
print('config    :', CONFIG)
print('recorded  : final {:.4f}  best {:.4f}'.format(
    ck['results']['final_acc'], ck['results']['best_acc']))
print('pinned at :', ck.get('pinned_commit'))
assert ck.get('pinned_commit') == PINNED_COMMIT, \
    'checkpoint was produced against a different upstream commit than this notebook builds'

In [ ]:
# ---- reproduce the exact train/test split ----
# The split must match the training run or 'the test set' means something different. It is
# deterministic given the same seed and stratification, both of which come from the checkpoint.
#
# The scaler is LOADED, not refitted. Refitting on a different split would shift the scaled
# space that the thermometer thresholds live in, and every threshold would then mean something
# slightly different -- a silent, plausible-looking corruption.
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

data = fetch_openml('hls4ml_lhc_jets_hlf', version=1, as_frame=True)
X = data.data.to_numpy(dtype=np.float32)
y = LabelEncoder().fit_transform(data.target.to_numpy())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=CONFIG['seed'], stratify=y)

mean = ck['scaler']['mean'].numpy()
scale = ck['scaler']['scale'].numpy()
X_test = ((X_test - mean) / scale).astype(np.float32)

print('test set  :', X_test.shape)
print('classes   :', ck['classes'])

In [ ]:
# ---- rebuild the model from the checkpoint ----
# Constructed exactly as the training notebook did, then state_dict loaded with strict=True so
# any structural mismatch is an error rather than a silently partial load.
from torch import nn

layers, in_size = [], 16 * CONFIG['thermometer_bits']
for i, width in enumerate(CONFIG['layers']):
    layers.append(dwn.LUTLayer(in_size, width, n=CONFIG['n'],
                               mapping=CONFIG['mapping'][i]))
    in_size = width
layers.append(dwn.GroupSum(k=CONFIG['num_classes'], tau=CONFIG['tau']))

model = nn.Sequential(*layers).cuda()
model.load_state_dict(ck['state_dict'], strict=True)
model.eval()
print('state_dict loaded, strict=True')

# Thermometer thresholds are LOADED, not refitted -- Thermometer is not an nn.Module, so they
# are not in the state_dict, but they are absolutely part of the model. Refitting them here
# would quietly change what every one of the 3200 threshold bits means.
THERMOMETERS = {'plain': dwn.Thermometer, 'gaussian': dwn.GaussianThermometer,
                'distributive': dwn.DistributiveThermometer}
thermometer = THERMOMETERS[ck['thermometer']['kind']](ck['thermometer']['num_bits'])
thermometer.thresholds = ck['thermometer']['thresholds']
print('thresholds:', tuple(thermometer.thresholds.shape), '(loaded, not refitted)')


In [ ]:
# ---- inference over the full test set ----
# Binarization and inference are fused per chunk so the 166k x 3200 binarized array is never
# materialised: as float32 that is 2.1 GB, and it is not needed for anything downstream.
import torch

X_test_t = torch.from_numpy(X_test)
preds = []
CHUNK = 5000

with torch.no_grad():
    for i in range(0, X_test_t.size(0), CHUNK):
        xb = thermometer.binarize(X_test_t[i:i+CHUNK]).flatten(start_dim=1).cuda()
        preds.append(model(xb).argmax(dim=1).cpu().numpy())
        del xb

pred_full = np.concatenate(preds)
acc = float((pred_full == y_test).mean())
print(f'inferred {pred_full.shape[0]} samples')
print(f'accuracy : {100*acc:.4f}%')

In [ ]:
# ---- VERIFY the reconstruction before trusting the output ----
# Two independent checks. Without these, a subtly wrong reconstruction (a refitted scaler, a
# different split) would produce a full test set that looks fine and quietly fails Gate 1b on
# the board, where it is far more expensive to diagnose.
ok = True

# 1. accuracy must match what the training run recorded
recorded = ck['results']['final_acc']
print(f'recorded final_acc : {100*recorded:.4f}%')
print(f'recomputed         : {100*acc:.4f}%')
if abs(acc - recorded) > 1e-6:
    print('  FAIL: accuracy differs -- the split, the scaler or the model does not match')
    ok = False
else:
    print('  OK: exact match')

# 2. the first 1000 predictions must match the committed test vectors, sample for sample.
#    This is the stronger check: it compares actual per-sample outputs, not an aggregate.
vec = sorted(glob.glob('/kaggle/input/**/*_testvectors.npz', recursive=True))
if vec:
    ref = np.load(vec[0])
    n = ref['pred'].shape[0]
    agree = int((pred_full[:n] == ref['pred']).sum())
    print(f'first {n} vs committed testvectors: {agree}/{n}')
    if agree != n:
        print('  FAIL: per-sample predictions differ from the verified reference')
        ok = False
    else:
        print('  OK: identical, so this is the same model Gate 1 was verified against')
else:
    print('testvectors.npz not in the dataset -- skipping the per-sample check.')
    print('Add it to the Kaggle Dataset for a much stronger guarantee.')

if not ok:
    raise SystemExit('Reconstruction checks FAILED -- do not use this output for Gate 1b.')
print()
print('Reconstruction verified.')

In [ ]:
# ---- save ----
# `pred` is the SOFTWARE MODEL's output, not ground truth, and the distinction matters: Gate 1b
# asks whether hardware reproduces the model to the sample. Disagreement with `pred` is a
# hardware bug; disagreement with `y` is just the model being wrong, which is already measured.
OUT = f'/kaggle/working/dwn_jsc_{RUN_NAME}_testset_full.npz'

np.savez_compressed(OUT, x_raw=X_test, y=y_test, pred=pred_full)

size_mb = os.path.getsize(OUT) / 1e6
print('wrote', OUT)
print(f'  {X_test.shape[0]} samples, {size_mb:.1f} MB')
print(f'  software accuracy {100*acc:.4f}%')
print()
print('Download it into training/artifacts/. It is gitignored -- regenerable by re-running')
print('this notebook, and too large to belong in git.')
print()
print('Then, with the board attached:')
print('  .venv\\Scripts\\python.exe scripts\\host.py --port COM4 --gate1b')

## What this does not do

It does **not** retrain, and it does not touch the checkpoint. Every number already measured
against that model — 108 core LUTs, 1519 encoder LUTs, 202 comparators, 147.1 MHz, the passing
Gate 1 runs — stays valid.

If you ever *do* want a retrained model, use `dwn_jsc_kaggle.ipynb` instead, and expect to
re-run `scripts/run_gate1.py` and `scripts/run_synth.py` afterwards: the RTL is regenerated from
whatever checkpoint you point it at, so the measurements move with it.